In [31]:
import os

print("Current folder:", os.getcwd())
print("Files here:", os.listdir())
print("Parent folder:", os.listdir(".."))

Current folder: /Users/stacychan/Query Analysis
Files here: ['.DS_Store', 'Reconciled_Data.csv', 'raw_data', 'Data_Preprocessing.ipynb', 'Reconciled_Data_with_Final_Fixed.csv', 'Reconciled_Final_Only_Fixed.csv']
Parent folder: ['.Rhistory', '.config', 'Music', '.zprofile.pysave', 'Query Analysis', '.condarc', '.docker', '.anyconnect', '.DS_Store', '.r', 'MPs', '.CFUserTextEncoding', '.xonshrc', 'anaconda_projects', '.subversion', '.zshrc', '.local', 'Pictures', '.zprofile', '.zsh_history', '.ipython', 'Desktop', 'Library', '.vpn', '.matplotlib', '.emulator_console_auth_token', '.spyder-py3', '.android', '.codex', 'Public', 'generative_agent', '.idlerc', '.tcshrc', '.cisco', '.RData', '.anaconda', 'Movies', 'Applications', '.gradle', '.Rapp.history', 'STAT385_FP_v2.R', '.Trash', '.jupyter', 'Documents', '.vscode', '.m2', '.bash_profile', 'Downloads', '.python_history', '.continuum', '.cache', '.gitconfig', 'STAT385_Final_Project.R', '.viminfo', '.zsh_sessions', '.conda']


In [32]:
import pandas as pd

df = pd.read_excel("raw_data/Query_Analysis.xlsx")

df.to_csv("Reconciled_Data.csv", index=False)


In [33]:
import pandas as pd
import numpy as np

file_path = "Reconciled_Data.csv"

# read raw csv
raw = pd.read_csv(file_path, header=None)

# actual data starts after the first 2 rows
df = raw.iloc[2:].copy().reset_index(drop=True)

# assign columns manually based on the file structure
df.columns = [
    "participant_no",
    "week_no",
    "query",

    "info_type_sophie",
    "info_type_ari",
    "info_type_bowen",
    "info_type_lexie",
    "info_type_agreement",
    "info_type_reconciliation",

    "task_sophie",
    "task_rachel",
    "task_bowen",
    "task_lexie",
    "task_agreement",
    "task_reconciliation",

    "goal_sophie",
    "goal_ari",
    "goal_bowen",
    "goal_lexie",
    "goal_agreement",
    "goal_reconciliation",
]

# turn empty strings into NA
df = df.replace(r"^\s*$", pd.NA, regex=True)

# fill merged-like blanks in participant/week
df["participant_no"] = df["participant_no"].ffill()
df["week_no"] = df["week_no"].ffill()

def is_yes(x):
    return pd.notna(x) and str(x).strip().upper() == "Y"

def fill_within_group(series):
    return series.ffill().bfill()

def first_mode(row_vals):
    vals = [str(v).strip() for v in row_vals if pd.notna(v) and str(v).strip() != ""]
    if not vals:
        return pd.NA
    return pd.Series(vals).value_counts().index[0]

group_keys = ["participant_no", "week_no", "query"]

# --------------------------------------------------
# 1) first fill reconciliation within repeated query rows
# --------------------------------------------------
for col in [
    "info_type_reconciliation",
    "task_reconciliation",
    "goal_reconciliation",
]:
    df[col] = df.groupby(group_keys, dropna=False)[col].transform(fill_within_group)

# --------------------------------------------------
# 2) if agreed = N but reconciliation is still blank,
#    fill reconciliation from coder columns
#    (majority vote among available coders)
# --------------------------------------------------

# info type fallback
mask = (~df["info_type_agreement"].apply(is_yes)) & (df["info_type_reconciliation"].isna())
df.loc[mask, "info_type_reconciliation"] = df.loc[
    mask,
    ["info_type_lexie", "info_type_bowen", "info_type_ari", "info_type_sophie"]
].apply(first_mode, axis=1)

# task fallback
mask = (~df["task_agreement"].apply(is_yes)) & (df["task_reconciliation"].isna())
df.loc[mask, "task_reconciliation"] = df.loc[
    mask,
    ["task_lexie", "task_bowen", "task_rachel", "task_sophie"]
].apply(first_mode, axis=1)

# goal fallback
mask = (~df["goal_agreement"].apply(is_yes)) & (df["goal_reconciliation"].isna())
df.loc[mask, "goal_reconciliation"] = df.loc[
    mask,
    ["goal_sophie", "goal_lexie", "goal_bowen", "goal_ari"]
].apply(first_mode, axis=1)

# --------------------------------------------------
# 3) apply your final rule
#    agreed = Y:
#       type/task from Lexie
#       goal from Sophie
#    agreed = N:
#       all from Reconciliation
# --------------------------------------------------
df["final_type"] = np.where(
    df["info_type_agreement"].apply(is_yes),
    df["info_type_lexie"],
    df["info_type_reconciliation"]
)

df["final_task"] = np.where(
    df["task_agreement"].apply(is_yes),
    df["task_lexie"],
    df["task_reconciliation"]
)

df["final_goal"] = np.where(
    df["goal_agreement"].apply(is_yes),
    df["goal_sophie"],
    df["goal_reconciliation"]
)

# --------------------------------------------------
# 4) last safety fill across repeated rows of same query
# --------------------------------------------------
for col in ["final_type", "final_task", "final_goal"]:
    df[col] = df.groupby(group_keys, dropna=False)[col].transform(fill_within_group)

# save full file
df.to_csv("Reconciled_Data_Full.csv", index=False)

# save final-only file
result = df[
    ["participant_no", "week_no", "query", "final_type", "final_task", "final_goal"]
].copy()

result.to_csv("Reconciled_Final.csv", index=False)

# check
print(result.isna().sum())
print(result.head(20))

/var/folders/_p/mv7dq2pn2_57jhgzwjp8dlrw0000gn/T/ipykernel_57627/1112842386.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.ffill().bfill()
/var/folders/_p/mv7dq2pn2_57jhgzwjp8dlrw0000gn/T/ipykernel_57627/1112842386.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.ffill().bfill()
/var/folders/_p/mv7dq2pn2_57jhgzwjp8dlrw0000gn/T/ipykernel_57627/1112842386.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=Fal

participant_no    0
week_no           0
query             0
final_type        0
final_task        0
final_goal        0
dtype: int64
   participant_no week_no                                     query  \
0             102  Week 1      foods to avoid while losing body fat   
1             102  Week 1      foods to avoid while losing body fat   
2             102  Week 1        foods to eat that help lose weight   
3             102  Week 1        foods to eat that help lose weight   
4             102  Week 1  vegan foods to eat that help lose weight   
5             102  Week 1                    is avocado food or bad   
6             102  Week 1         is avocado too much fat with oil'   
7             102  Week 1        is it ok to eat chips occasionally   
8             102  Week 1        is it ok to eat chips occasionally   
9             102  Week 1   is it ok to eat chips occasionally [10]   
10            102  Week 1     how much water should you drink a day   
11            1